In [1]:
import os
import findspark

os.environ["SPARK_HOME"] = "/workspaces/Stream_Analytics-Group_7/spark-4.0.1-bin-hadoop3"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

findspark.init()

import pyspark
print("PySpark version:", pyspark.__version__)
print("SPARK_HOME:", os.environ["SPARK_HOME"])
print("JAVA_HOME:", os.environ["JAVA_HOME"])

PySpark version: 4.0.1
SPARK_HOME: /workspaces/Stream_Analytics-Group_7/spark-4.0.1-bin-hadoop3
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("group7-gold-outputs")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-azure:3.3.4,com.microsoft.azure:azure-storage:8.6.6"
    )
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

print("Spark version:", spark.version)

:: loading settings :: url = jar:file:/workspaces/Stream_Analytics-Group_7/spark-4.0.1-bin-hadoop3/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/codespace/.ivy2.5.2/cache
The jars for the packages stored in: /home/codespace/.ivy2.5.2/jars
org.apache.hadoop#hadoop-azure added as a dependency
com.microsoft.azure#azure-storage added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b19052cf-a771-428c-a08a-b8815f0c2f35;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-guava;1.1.1 in central
	found org.eclipse.jetty#jetty-util-ajax;9.4.43.v20210629 in central
	found org.eclipse.jetty#jetty-util;9.4.43.v202106

Spark version: 4.0.1


In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv

cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd

env_path = repo_root / ".env"
load_dotenv(env_path, override=True)

print("Repo root:", repo_root)
print("ENV exists:", env_path.exists())
print("STORAGE_ACCOUNT_NAME:", os.getenv("STORAGE_ACCOUNT_NAME"))
print("STORAGE_CONTAINER_NAME:", os.getenv("STORAGE_CONTAINER_NAME"))
print("STORAGE_ACCOUNT_ACCESS_KEY loaded:", bool(os.getenv("STORAGE_ACCOUNT_ACCESS_KEY")))

Repo root: /workspaces/Stream_Analytics-Group_7
ENV exists: True
STORAGE_ACCOUNT_NAME: iesstsabbadbaa
STORAGE_CONTAINER_NAME: group7
STORAGE_ACCOUNT_ACCESS_KEY loaded: True


In [5]:
bronze_orders_path = repo_root / "data" / "bronze" / "orders"
bronze_delivery_path = repo_root / "data" / "bronze" / "delivery"

orders = spark.read.parquet(str(bronze_orders_path))
delivery = spark.read.parquet(str(bronze_delivery_path))

print("Orders:", orders.count())
print("Delivery:", delivery.count())

Orders: 94
Delivery: 738


In [6]:
from pyspark.sql.functions import (
    col, when, count, sum, avg, lit, window, unix_timestamp, expr
)

In [7]:
orders_silver = (
    orders
    .select(
        "event_id", "order_id", "event_type", "restaurant_id", "zone_id",
        "courier_id", "event_time", "ingest_time", "order_value",
        "is_promo", "estimated_delivery_minutes", "version"
    )
    .filter(col("event_id").isNotNull())
    .filter(col("order_id").isNotNull())
    .filter(col("zone_id").isNotNull())
    .filter(col("event_time").isNotNull())
)

orders_silver.show(5, truncate=False)

+------------------------------------+------------------------------------+----------------+-------------+-------+----------+-----------------------+-----------------------+-----------+--------+--------------------------+-------+
|event_id                            |order_id                            |event_type      |restaurant_id|zone_id|courier_id|event_time             |ingest_time            |order_value|is_promo|estimated_delivery_minutes|version|
+------------------------------------+------------------------------------+----------------+-------------+-------+----------+-----------------------+-----------------------+-----------+--------+--------------------------+-------+
|64dce1a6-2081-43d3-a08e-37e1a9b9aa4d|f750a99f-fa2c-45e0-b0c3-1da8d120fb87|prep_started    |rest_20      |east   |NULL      |2026-04-16 20:19:24.093|2026-04-16 20:36:46.102|37.25      |false   |36                        |1      |
|6dc3ffd9-5d22-43a3-b084-c78e20d54106|f750a99f-fa2c-45e0-b0c3-1da8d120fb87|picke

In [8]:
delivery_silver = (
    delivery
    .select(
        "event_id", "courier_id", "event_type", "courier_status", "zone_id",
        "order_id", "event_time", "ingest_time", "latitude", "longitude", "version"
    )
    .filter(col("event_id").isNotNull())
    .filter(col("courier_id").isNotNull())
    .filter(col("zone_id").isNotNull())
    .filter(col("event_time").isNotNull())
)

delivery_silver.show(5, truncate=False)

+------------------------------------+----------+---------------+---------------------+-------+------------------------------------+-----------------------+-----------------------+---------+---------+-------+
|event_id                            |courier_id|event_type     |courier_status       |zone_id|order_id                            |event_time             |ingest_time            |latitude |longitude|version|
+------------------------------------+----------+---------------+---------------------+-------+------------------------------------+-----------------------+-----------------------+---------+---------+-------+
|076e8d68-8be1-47e8-bcf6-c893b137cb42|courier_0 |status_change  |idle                 |south  |NULL                                |2026-03-06 12:26:03.427|2026-03-06 12:26:03.427|0.0      |0.0      |1      |
|440cf909-3515-4dc0-b0a7-af47eb8a9ad7|courier_0 |status_change  |waiting_at_restaurant|south  |11e87881-98f0-4e81-b0c0-9feb400e21c9|2026-03-06 12:26:33.427|2026-03-

In [9]:
gold_orders_kpi_5m = (
    orders_silver
    .groupBy(window("event_time", "5 minutes"), "zone_id")
    .agg(
        count("*").alias("orders_total"),
        sum(when(col("event_type") == "cancelled", 1).otherwise(0)).alias("cancelled_orders"),
        sum(when(col("event_type") == "delivered", 1).otherwise(0)).alias("delivered_orders"),
        avg("order_value").alias("avg_order_value")
    )
    .withColumn("cancel_rate", col("cancelled_orders") / (col("orders_total") + lit(1)))
)

gold_orders_kpi_5m.show(truncate=False)

+------------------------------------------+-------+------------+----------------+----------------+------------------+-----------+
|window                                    |zone_id|orders_total|cancelled_orders|delivered_orders|avg_order_value   |cancel_rate|
+------------------------------------------+-------+------------+----------------+----------------+------------------+-----------+
|{2026-04-16 20:15:00, 2026-04-16 20:20:00}|west   |10          |0               |0               |23.822            |0.0        |
|{2026-04-16 20:25:00, 2026-04-16 20:30:00}|south  |3           |0               |0               |21.723333333333333|0.0        |
|{2026-04-16 20:35:00, 2026-04-16 20:40:00}|center |13          |0               |0               |28.01615384615385 |0.0        |
|{2026-04-16 20:55:00, 2026-04-16 21:00:00}|south  |1           |0               |1               |29.74             |0.0        |
|{2026-04-16 20:20:00, 2026-04-16 20:25:00}|west   |3           |2               |0

In [10]:
waiting_orders = (
    orders_silver
    .filter(col("event_type") == "created")
    .groupBy("zone_id")
    .agg(count("*").alias("waiting_orders"))
)

available_delivery = (
    delivery_silver
    .filter(col("event_type") == "status_change")
    .groupBy("zone_id")
    .agg(count("*").alias("available_couriers"))
)

gold_demand_supply_zone = (
    waiting_orders
    .join(available_delivery, on="zone_id", how="outer")
    .fillna(0)
    .withColumn("demand_supply_ratio", col("waiting_orders") / (col("available_couriers") + lit(1)))
)

gold_demand_supply_zone.show(truncate=False)

+-------+--------------+------------------+--------------------+
|zone_id|waiting_orders|available_couriers|demand_supply_ratio |
+-------+--------------+------------------+--------------------+
|center |8             |56                |0.14035087719298245 |
|east   |3             |66                |0.04477611940298507 |
|north  |0             |54                |0.0                 |
|south  |2             |42                |0.046511627906976744|
|west   |5             |58                |0.0847457627118644  |
+-------+--------------+------------------+--------------------+



In [11]:
orders_created = (
    orders_silver
    .filter(col("event_type") == "created")
    .select(col("order_id"), col("zone_id"), col("event_time").alias("created_time"))
)

orders_delivered = (
    orders_silver
    .filter(col("event_type") == "delivered")
    .select(col("order_id"), col("event_time").alias("delivered_time"))
)

delivery_time_per_order = (
    orders_created
    .join(orders_delivered, on="order_id", how="inner")
    .withColumn(
        "delivery_minutes",
        (unix_timestamp("delivered_time") - unix_timestamp("created_time")) / 60.0
    )
    .filter(col("delivery_minutes") >= 0)
    .filter(col("delivery_minutes") <= 180)
)

zone_delivery_stats = (
    delivery_time_per_order
    .groupBy("zone_id")
    .agg(
        avg("delivery_minutes").alias("avg_delivery_minutes"),
        count("*").alias("delivered_orders"),
        expr("percentile_approx(delivery_minutes, 0.95)").alias("p95_delivery_minutes")
    )
)

gold_delivery_anomalies = (
    zone_delivery_stats
    .withColumn(
        "anomaly_flag",
        when((col("avg_delivery_minutes") > 45) | (col("p95_delivery_minutes") > 60), 1).otherwise(0)
    )
)

gold_delivery_anomalies.show(truncate=False)

+-------+--------------------+----------------+--------------------+------------+
|zone_id|avg_delivery_minutes|delivered_orders|p95_delivery_minutes|anomaly_flag|
+-------+--------------------+----------------+--------------------+------------+
|east   |31.666666666666668  |3               |37.0                |0           |
|west   |36.666666666666664  |3               |46.0                |0           |
|center |39.57142857142857   |7               |45.0                |0           |
|south  |35.5                |2               |40.0                |0           |
+-------+--------------------+----------------+--------------------+------------+



In [12]:
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")
storage_container_name = os.getenv("STORAGE_CONTAINER_NAME")
storage_account_key = os.getenv("STORAGE_ACCOUNT_ACCESS_KEY")

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    storage_account_key
)

blob_base = f"wasbs://{storage_container_name}@{storage_account_name}.blob.core.windows.net"

gold_orders_blob_path = f"{blob_base}/gold/orders_kpi_5m"
gold_demand_supply_blob_path = f"{blob_base}/gold/demand_supply_zone"
gold_anomalies_blob_path = f"{blob_base}/gold/delivery_anomalies"

print(blob_base)
print(gold_orders_blob_path)
print(gold_demand_supply_blob_path)
print(gold_anomalies_blob_path)

wasbs://group7@iesstsabbadbaa.blob.core.windows.net
wasbs://group7@iesstsabbadbaa.blob.core.windows.net/gold/orders_kpi_5m
wasbs://group7@iesstsabbadbaa.blob.core.windows.net/gold/demand_supply_zone
wasbs://group7@iesstsabbadbaa.blob.core.windows.net/gold/delivery_anomalies


In [13]:
gold_orders_kpi_5m.write.mode("overwrite").parquet(gold_orders_blob_path)
gold_demand_supply_zone.write.mode("overwrite").parquet(gold_demand_supply_blob_path)
gold_delivery_anomalies.write.mode("overwrite").parquet(gold_anomalies_blob_path)

print("Gold outputs written to Azure Blob")

26/04/19 15:35:32 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-azure-file-system.properties,hadoop-metrics2.properties
26/04/19 15:35:34 WARN AzureFileSystemThreadPoolExecutor: Disabling threads for Rename operation as thread count 0 is <= 1
26/04/19 15:35:34 WARN AzureFileSystemThreadPoolExecutor: Disabling threads for Delete operation as thread count 0 is <= 1
26/04/19 15:35:36 WARN AzureFileSystemThreadPoolExecutor: Disabling threads for Rename operation as thread count 0 is <= 1
26/04/19 15:35:37 WARN AzureFileSystemThreadPoolExecutor: Disabling threads for Delete operation as thread count 0 is <= 1
26/04/19 15:35:39 WARN AzureFileSystemThreadPoolExecutor: Disabling threads for Rename operation as thread count 0 is <= 1
26/04/19 15:35:39 WARN AzureFileSystemThreadPoolExecutor: Disabling threads for Delete operation as thread count 0 is <= 1


Gold outputs written to Azure Blob


In [14]:
blob_orders = spark.read.parquet(gold_orders_blob_path)
blob_demand_supply = spark.read.parquet(gold_demand_supply_blob_path)
blob_anomalies = spark.read.parquet(gold_anomalies_blob_path)

print("Blob orders rows:", blob_orders.count())
print("Blob demand_supply rows:", blob_demand_supply.count())
print("Blob anomalies rows:", blob_anomalies.count())

blob_orders.show(5, truncate=False)
blob_demand_supply.show(5, truncate=False)
blob_anomalies.show(5, truncate=False)

Blob orders rows: 25
Blob demand_supply rows: 5
Blob anomalies rows: 4
+------------------------------------------+-------+------------+----------------+----------------+------------------+-----------+
|window                                    |zone_id|orders_total|cancelled_orders|delivered_orders|avg_order_value   |cancel_rate|
+------------------------------------------+-------+------------+----------------+----------------+------------------+-----------+
|{2026-04-16 20:15:00, 2026-04-16 20:20:00}|west   |10          |0               |0               |23.822            |0.0        |
|{2026-04-16 20:25:00, 2026-04-16 20:30:00}|south  |3           |0               |0               |21.723333333333333|0.0        |
|{2026-04-16 20:35:00, 2026-04-16 20:40:00}|center |13          |0               |0               |28.01615384615385 |0.0        |
|{2026-04-16 20:55:00, 2026-04-16 21:00:00}|south  |1           |0               |1               |29.74             |0.0        |
|{2026-04-16